# 空间转录组质量控制实战

> 本 Notebook 对应文章：`005_S4_空间转录组质量控制实战[通用][QC][进阶].md`
>
> 目标：逐步执行文章中的所有代码，验证可行性

## 环境准备

首次运行时，请先安装依赖包

In [ ]:
# 安装依赖（首次运行时取消注释）
# !pip install scanpy squidpy matplotlib

In [ ]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white')

### Step 1: 加载数据并计算 QC 指标

In [ ]:
# Load Visium data
adata = sc.read_visium(
    path='data/visium_sample/',
    count_file='filtered_feature_bc_matrix.h5'
)

# Add spatial information
sq.im.calculate_image_features(
    adata,
    features='summary',
    key_added='image_features'
)

# Calculate QC metrics
# Identify mitochondrial genes (species-specific)
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # Human
# adata.var['mt'] = adata.var_names.str.startswith('mt-')  # Mouse

# Calculate per-spot QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print(f"Total spots: {adata.n_obs}")
print(f"Total genes: {adata.n_vars}")
print(f"Spots in tissue: {adata.obs['in_tissue'].sum()}")

**输出示例**：

### Step 2: 诊断图 - 理解数据分布

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Total UMI counts distribution
ax = axes[0, 0]
sns.histplot(
    data=adata.obs,
    x='total_counts',
    hue='in_tissue',
    bins=50,
    ax=ax,
    kde=True
)
ax.set_xlabel('Total UMI Counts')
ax.set_ylabel('Number of Spots')
ax.set_title('UMI Count Distribution')
ax.axvline(x=500, color='red', linestyle='--', label='Threshold: 500')
ax.legend()

# 2. Number of genes detected
ax = axes[0, 1]
sns.histplot(
    data=adata.obs,
    x='n_genes_by_counts',
    hue='in_tissue',
    bins=50,
    ax=ax,
    kde=True
)
ax.set_xlabel('Number of Genes Detected')
ax.set_ylabel('Number of Spots')
ax.set_title('Gene Detection Distribution')
ax.axvline(x=200, color='red', linestyle='--', label='Threshold: 200')
ax.legend()

# 3. Mitochondrial gene percentage
ax = axes[0, 2]
sns.histplot(
    data=adata.obs,
    x='pct_counts_mt',
    hue='in_tissue',
    bins=50,
    ax=ax,
    kde=True
)
ax.set_xlabel('Mitochondrial Gene Percentage (%)')
ax.set_ylabel('Number of Spots')
ax.set_title('MT Gene Distribution')
ax.axvline(x=20, color='red', linestyle='--', label='Threshold: 20%')
ax.legend()

# 4. Spatial distribution of total counts
ax = axes[1, 0]
sq.pl.spatial_scatter(
    adata,
    color='total_counts',
    size=1.5,
    ax=ax,
    title='Spatial Distribution: Total UMI'
)

# 5. Spatial distribution of gene counts
ax = axes[1, 1]
sq.pl.spatial_scatter(
    adata,
    color='n_genes_by_counts',
    size=1.5,
    ax=ax,
    title='Spatial Distribution: Gene Counts'
)

# 6. Spatial distribution of MT percentage
ax = axes[1, 2]
sq.pl.spatial_scatter(
    adata,
    color='pct_counts_mt',
    size=1.5,
    ax=ax,
    title='Spatial Distribution: MT Percentage'
)

plt.tight_layout()
plt.savefig('qc_diagnostic_plots.png', dpi=300, bbox_inches='tight')
plt.show()

**关键观察点**：
- **组织内 vs 组织外**：组织外 spot 的 UMI 数应该显著更低
- **空间连续性**：相邻 spot 的指标应该相似，突变点可能是异常值
- **线粒体比例**：边缘区域可能因组织损伤而升高


### Step 3: 负对照验证 - 组织外 spot 的指标分布

In [ ]:
# Compare in-tissue vs out-of-tissue spots
in_tissue = adata.obs[adata.obs['in_tissue'] == 1]
out_tissue = adata.obs[adata.obs['in_tissue'] == 0]

print("=== QC Metrics Comparison ===")
print(f"\nIn-tissue spots (n={len(in_tissue)}):")
print(f"  Total UMI: {in_tissue['total_counts'].median():.0f} (median)")
print(f"  Genes detected: {in_tissue['n_genes_by_counts'].median():.0f} (median)")
print(f"  MT percentage: {in_tissue['pct_counts_mt'].median():.2f}% (median)")

print(f"\nOut-of-tissue spots (n={len(out_tissue)}):")
print(f"  Total UMI: {out_tissue['total_counts'].median():.0f} (median)")
print(f"  Genes detected: {out_tissue['n_genes_by_counts'].median():.0f} (median)")
print(f"  MT percentage: {out_tissue['pct_counts_mt'].median():.2f}% (median)")

# Statistical test
from scipy.stats import mannwhitneyu

stat, pval = mannwhitneyu(
    in_tissue['total_counts'],
    out_tissue['total_counts'],
    alternative='greater'
)
print(f"\nMann-Whitney U test (total_counts): p-value = {pval:.2e}")

**输出示例**：

**边界声明 (Boundary Statement)**：
- ✅ 能说明：组织外 spot 的 UMI 数显著低于组织内，验证了图像分割的准确性
- ❌ 不能说明：所有组织内 spot 都是高质量的（仍需进一步过滤）

### Step 4: 设置过滤阈值 - 数据驱动的决策

In [ ]:
# Calculate percentiles for threshold selection
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
metrics = ['total_counts', 'n_genes_by_counts', 'pct_counts_mt']

print("=== Percentile Distribution (In-tissue spots only) ===\n")
for metric in metrics:
    print(f"{metric}:")
    values = np.percentile(in_tissue[metric], percentiles)
    for p, v in zip(percentiles, values):
        print(f"  {p}th percentile: {v:.2f}")
    print()

# Recommended thresholds based on data distribution
# Lower bound: exclude bottom 1-5% (likely technical failures)
# Upper bound: exclude top 1-5% (likely doublets or artifacts)
# MT percentage: exclude top 5-10% (likely damaged cells)

threshold_lower_counts = np.percentile(in_tissue['total_counts'], 1)
threshold_lower_genes = np.percentile(in_tissue['n_genes_by_counts'], 1)
threshold_upper_mt = np.percentile(in_tissue['pct_counts_mt'], 95)

print("=== Recommended Thresholds ===")
print(f"Min total UMI: {threshold_lower_counts:.0f}")
print(f"Min genes detected: {threshold_lower_genes:.0f}")
print(f"Max MT percentage: {threshold_upper_mt:.2f}%")

**输出示例**：

### Step 5: 执行过滤并验证

In [ ]:
# Store original counts for comparison
n_spots_before = adata.n_obs
n_genes_before = adata.n_vars

# Filter spots
adata = adata[adata.obs['in_tissue'] == 1].copy()
print(f"After removing out-of-tissue spots: {adata.n_obs} spots")

# Apply QC thresholds
sc.pp.filter_cells(adata, min_counts=threshold_lower_counts)
sc.pp.filter_cells(adata, min_genes=threshold_lower_genes)
adata = adata[adata.obs['pct_counts_mt'] < threshold_upper_mt].copy()

print(f"After QC filtering: {adata.n_obs} spots")
print(f"Spots removed: {n_spots_before - adata.n_obs} ({(n_spots_before - adata.n_obs) / n_spots_before * 100:.1f}%)")

# Filter genes (expressed in at least 3 spots)
sc.pp.filter_genes(adata, min_cells=3)
print(f"After gene filtering: {adata.n_vars} genes")
print(f"Genes removed: {n_genes_before - adata.n_vars} ({(n_genes_before - adata.n_vars) / n_genes_before * 100:.1f}%)")

# Recalculate QC metrics after filtering
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

**输出示例**：

### Step 6: 敏感性分析 - 阈值稳健性测试

In [ ]:
# Test different threshold combinations
threshold_scenarios = [
    {'name': 'Lenient', 'min_counts': 300, 'min_genes': 150, 'max_mt': 15},
    {'name': 'Moderate', 'min_counts': 500, 'min_genes': 200, 'max_mt': 12},
    {'name': 'Stringent', 'min_counts': 1000, 'min_genes': 400, 'max_mt': 10},
]

# Reload original data for comparison
adata_original = sc.read_visium(
    path='data/visium_sample/',
    count_file='filtered_feature_bc_matrix.h5'
)
adata_original.var['mt'] = adata_original.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(
    adata_original,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

results = []
for scenario in threshold_scenarios:
    adata_test = adata_original[adata_original.obs['in_tissue'] == 1].copy()
    
    # Apply filters
    adata_test = adata_test[
        (adata_test.obs['total_counts'] >= scenario['min_counts']) &
        (adata_test.obs['n_genes_by_counts'] >= scenario['min_genes']) &
        (adata_test.obs['pct_counts_mt'] < scenario['max_mt'])
    ].copy()
    
    results.append({
        'Scenario': scenario['name'],
        'Spots Retained': adata_test.n_obs,
        'Retention Rate': f"{adata_test.n_obs / adata_original.obs['in_tissue'].sum() * 100:.1f}%",
        'Median UMI': int(adata_test.obs['total_counts'].median()),
        'Median Genes': int(adata_test.obs['n_genes_by_counts'].median()),
        'Median MT%': f"{adata_test.obs['pct_counts_mt'].median():.2f}"
    })

results_df = pd.DataFrame(results)
print("\n=== Sensitivity Analysis: Threshold Impact ===")
print(results_df.to_string(index=False))

**输出示例**：

**解读**：
- Lenient 保留了 98.7% 的组织内 spot，但可能包含低质量数据
- Stringent 只保留 86.2%，可能过度过滤了边缘区域的真实信号
- Moderate 是平衡选择，保留 95.9% 的 spot，同时确保数据质量


### Step 7: 后验证 - 确认过滤效果

In [ ]:
# Visualize before/after comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before filtering (use original data)
ax = axes[0]
sq.pl.spatial_scatter(
    adata_original[adata_original.obs['in_tissue'] == 1],
    color='total_counts',
    size=1.5,
    ax=ax,
    title='Before QC Filtering',
    vmin=0,
    vmax=20000
)

# After filtering
ax = axes[1]
sq.pl.spatial_scatter(
    adata,
    color='total_counts',
    size=1.5,
    ax=ax,
    title='After QC Filtering',
    vmin=0,
    vmax=20000
)

plt.tight_layout()
plt.savefig('qc_before_after_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("\n=== QC Summary ===")
print(f"Original spots (in-tissue): {adata_original.obs['in_tissue'].sum()}")
print(f"Retained spots: {adata.n_obs}")
print(f"Retention rate: {adata.n_obs / adata_original.obs['in_tissue'].sum() * 100:.1f}%")
print(f"\nMedian UMI (before): {adata_original[adata_original.obs['in_tissue'] == 1].obs['total_counts'].median():.0f}")
print(f"Median UMI (after): {adata.obs['total_counts'].median():.0f}")
print(f"\nMedian MT% (before): {adata_original[adata_original.obs['in_tissue'] == 1].obs['pct_counts_mt'].median():.2f}%")
print(f"Median MT% (after): {adata.obs['pct_counts_mt'].median():.2f}%")

---

## 特殊场景的 QC 策略

### 场景 1: Visium HD 数据

Visium HD (High Definition) 是 10x Genomics 推出的高分辨率空间转录组平台，提供 2 µm 和 8 µm 两种分辨率，QC 策略需要调整：

In [ ]:
# For 8 µm bin (recommended for most analyses)
# Similar to standard Visium, but expect lower UMI per bin
threshold_lower_counts_hd8 = 200  # Lower than Visium (55 µm)
threshold_lower_genes_hd8 = 100

# For 2 µm bin (single-cell resolution)
# Expect much lower UMI per bin, similar to scRNA-seq
threshold_lower_counts_hd2 = 50
threshold_lower_genes_hd2 = 30

# Memory management for HD data
import gc
adata_hd = sc.read_h5ad('visium_hd_8um.h5ad', backed='r')  # Use backed mode
adata_hd = adata_hd[adata_hd.obs['in_tissue'] == 1].to_memory()  # Load subset
gc.collect()

**关键差异**：
- HD 数据的 spot/bin 更小，UMI 数更低
- 2 µm bin 接近单细胞分辨率，需要参考 scRNA-seq 的 QC 标准
- 内存占用大，建议使用 backed mode 或分块处理

### 场景 2: 成像型平台 (Xenium, CosMx, MERFISH)

成像型平台的 QC 重点不同：

In [ ]:
# For imaging-based platforms
# Focus on transcript detection quality, not UMI counts

# Key metrics:
# 1. Transcript counts per cell
# 2. Decoding quality score
# 3. Cell segmentation quality (nucleus vs cytoplasm ratio)

# Example for Xenium data
adata_xenium = sc.read_h5ad('xenium_data.h5ad')

# Check decoding quality
adata_xenium.obs['qv_mean'] = adata_xenium.obs['qv'].mean(axis=1)  # Quality value
adata_xenium = adata_xenium[adata_xenium.obs['qv_mean'] > 20].copy()  # Phred score > 20

# Check cell area (remove abnormally large/small cells)
adata_xenium = adata_xenium[
    (adata_xenium.obs['cell_area'] > 20) &
    (adata_xenium.obs['cell_area'] < 500)
].copy()

# Check nucleus/cytoplasm ratio
adata_xenium.obs['nucleus_ratio'] = (
    adata_xenium.obs['nucleus_area'] / adata_xenium.obs['cell_area']
)
adata_xenium = adata_xenium[
    (adata_xenium.obs['nucleus_ratio'] > 0.1) &
    (adata_xenium.obs['nucleus_ratio'] < 0.8)
].copy()

### 场景 3: 多样本批次效应检查

当分析多个样本时，需要检查批次效应 (Batch Effect)：

In [ ]:
# Load multiple samples
samples = ['sample1', 'sample2', 'sample3']
adatas = []

for sample in samples:
    adata_sample = sc.read_visium(f'data/{sample}/')
    adata_sample.obs['sample_id'] = sample
    adatas.append(adata_sample)

# Concatenate
adata_multi = sc.concat(adatas, label='batch', keys=samples)

# Compare QC metrics across batches
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['total_counts', 'n_genes_by_counts', 'pct_counts_mt']
titles = ['Total UMI Counts', 'Number of Genes', 'MT Percentage']

for ax, metric, title in zip(axes, metrics, titles):
    sns.violinplot(
        data=adata_multi.obs,
        x='batch',
        y=metric,
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel('Sample')

plt.tight_layout()
plt.savefig('batch_qc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical test for batch differences
from scipy.stats import kruskal

for metric in metrics:
    groups = [adata_multi.obs[adata_multi.obs['batch'] == s][metric] for s in samples]
    stat, pval = kruskal(*groups)
    print(f"{metric}: Kruskal-Wallis p-value = {pval:.2e}")

**边界声明**：
- ✅ 能说明：不同样本间的技术变异程度
- ❌ 不能说明：这些差异是否需要批次校正（需要结合生物学背景判断）

---

## 验收清单：你的 QC 是否完整？

在进入下游分析前，确认以下检查项：

### 数据层面
- [ ] 计算了三大核心指标：total_counts, n_genes_by_counts, pct_counts_mt
- [ ] 绘制了分布图（直方图 + 空间分布图）
- [ ] 比较了组织内外 spot 的指标差异（负对照）
- [ ] 设置了数据驱动的过滤阈值（基于百分位数）
- [ ] 执行了过滤并记录了保留率

### 证据链层面
- [ ] 诊断图：展示了过滤前后的对比
- [ ] 对照：验证了组织外 spot 的低质量特征
- [ ] 敏感性：测试了不同阈值的影响
- [ ] 边界声明：明确了 QC 能解决和不能解决的问题

### 可复现层面
- [ ] 设置了随机种子
- [ ] 记录了所有阈值参数
- [ ] 保存了过滤后的数据对象
- [ ] 导出了 QC 报告图表

### 特殊场景
- [ ] 如果是 HD 数据，调整了阈值并考虑了内存管理
- [ ] 如果是成像型数据，检查了解码质量和细胞分割
- [ ] 如果是多样本，检查了批次效应

---

## 常见的坑与解决方案

### 坑 1: 过度过滤导致空间结构丢失

**现象**：过滤后某些组织区域完全消失，如肿瘤边缘或坏死区域。

**原因**：这些区域的细胞本身就处于应激状态，UMI 数较低、线粒体比例较高。

**解决方案**：

In [ ]:
# 分区域设置阈值
# 1. 先做初步聚类
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.leiden(adata, resolution=0.5)

# 2. 检查每个 cluster 的 QC 指标分布
cluster_qc = adata.obs.groupby('leiden')[['total_counts', 'pct_counts_mt']].median()
print(cluster_qc)

# 3. 对低质量 cluster 单独评估
# 如果某个 cluster 在空间上连续且有生物学意义（如坏死区），考虑保留

### 坑 2: 线粒体比例阈值设置不当

**现象**：某些组织类型（如心肌、肝脏）的线粒体基因本身就高表达。

**原因**：线粒体比例的"正常范围"因组织类型而异。

**解决方案**：

In [ ]:
# 参考文献或公开数据集的同类组织
# 例如：心肌组织的 MT% 可能在 10-20% 是正常的
# 而脑组织通常 < 5%

# 使用组织特异性阈值
tissue_type = 'heart'  # or 'brain', 'liver', etc.
mt_thresholds = {
    'brain': 5,
    'heart': 20,
    'liver': 15,
    'tumor': 12
}
threshold_upper_mt = mt_thresholds.get(tissue_type, 10)  # Default: 10%

### 坑 3: 忽略空间连续性

**现象**：过滤后的数据在空间上出现"孤岛"或"空洞"。

**原因**：仅基于单个 spot 的指标过滤，没有考虑邻域 (Neighborhood) 信息。

**解决方案**：

In [ ]:
# 使用空间平滑后的指标
sq.gr.spatial_neighbors(adata, coord_type='grid')

# Calculate neighborhood-averaged metrics
from scipy.sparse import csr_matrix
adj_matrix = adata.obsp['spatial_connectivities']

for metric in ['total_counts', 'n_genes_by_counts']:
    # Smooth metric using spatial neighbors
    smoothed = adj_matrix @ adata.obs[metric].values
    smoothed = smoothed / adj_matrix.sum(axis=1).A1  # Normalize by neighbor count
    adata.obs[f'{metric}_smoothed'] = smoothed

# Use smoothed metrics for filtering
adata = adata[adata.obs['total_counts_smoothed'] > threshold].copy()

### 坑 4: 基因过滤过于激进

**现象**：过滤掉了低表达但生物学重要的基因（如转录因子）。

**原因**：`min_cells=3` 的阈值可能对大数据集太宽松，对小数据集太严格。

**解决方案**：

In [ ]:
# 动态调整基因过滤阈值
n_spots = adata.n_obs
min_cells_threshold = max(3, int(n_spots * 0.001))  # At least 0.1% of spots

sc.pp.filter_genes(adata, min_cells=min_cells_threshold)

# 或者保留特定基因列表（如已知的标志基因）
# EPCAM: 上皮细胞粘附分子, VIM: 波形蛋白（间质标志）
marker_genes = ['CD3D', 'CD8A', 'CD4', 'EPCAM', 'VIM']  # Example
genes_to_keep = adata.var_names.isin(marker_genes)
genes_pass_filter = adata.var['n_cells'] >= min_cells_threshold

adata = adata[:, genes_to_keep | genes_pass_filter].copy()

---

## 下一步：归一化 (Normalization) 与批次校正

完成 QC 后，数据已经清理干净，但不同 spot 之间的测序深度仍然不同。下一篇文章将介绍：

1. **归一化方法选择**：
   - Library size normalization (Scanpy)
   - SCTransform (Seurat 移植)
   - Pearson residuals (新兴方法)

2. **批次效应校正**：
   - Harmony
   - scVI
   - ComBat

3. **空间特异性考虑**：
   - 如何在归一化时保留空间信息？
   - 邻域信息是否应该参与归一化？

**预告**：归一化不仅影响下游聚类，还会影响空间可变基因的检测。选错方法可能让你错过关键的空间模式。

---

## 证据链总结

本文通过完整的证据链展示了 QC 流程的可靠性：

1. **诊断图**：
   - 分布图展示了数据的整体质量
   - 空间分布图揭示了质量指标的空间模式
   - 前后对比图验证了过滤效果

2. **负对照**：
   - 组织外 spot 的低质量特征验证了阈值的合理性
   - Mann-Whitney U 检验提供了统计显著性

3. **敏感性分析**：
   - 三种阈值场景展示了过滤策略的稳健性
   - 保留率和中位数指标帮助选择最优方案

4. **边界声明**：
   - QC 能清除技术噪音，但不能修正生物学变异
   - 过滤阈值需要根据组织类型和研究目标调整
   - 空间连续性检查是必要的补充验证

**最终交付**：一个经过严格质控、可用于下游分析的 AnnData 对象，以及完整的 QC 报告图表。

---

## 参考资源

- Scanpy 官方文档中的 QC 教程
- Squidpy 空间分析文档
- 10x Genomics Visium 官方 QC 指南